<a href="https://colab.research.google.com/github/Fa-Alsuqayri/Agentic-AI-System-Assignments/blob/main/Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Faisal Alsuqayri

---

##Assignment 5
###Use case: Using an article on Generative Adversarial Networks (GANs) as the new data source

In [12]:
# %pip install langchain langchain-core "langchain[openai]" langchain-text-splitters langchain-community bs4 langchain-huggingface sentence-transformers

In [3]:
import os
import bs4
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
from google.colab import userdata

In [7]:
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [13]:
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)
vector_store = InMemoryVectorStore(embeddings)

bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Generative_adversarial_network",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

# 3. Split and Store
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
vector_store.add_documents(documents=all_splits)

print(f"Indexed {len(all_splits)} document chunks.")

In [10]:
from typing import Literal
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n"
        f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model_nemotron3_nano,
    tools=[retrieve_context],
    system_prompt=(
        "You have access to a tool that retrieves context from a technical article. "
        "Use the tool to help answer user queries. Break down complex questions and search step-by-step if necessary."
    )
)

multi_step_query = (
    "What are the two main neural networks that make up a GAN?\n\n"
    "Once you have the answer, look up what the specific training objective is for the discriminator network."
)

result = agent.invoke({"messages": [HumanMessage(multi_step_query)]})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What are the two main neural networks that make up a GAN?

Once you have the answer, look up what the specific training objective is for the discriminator network.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_12d06b166367475485b9d36b)
 Call ID: call_12d06b166367475485b9d36b
  Args:
    query: What are the two main neural networks that make up a GAN?
================================= Tool Message =================================
Name: retrieve_context


================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_cad80716089345b49fa56e01)
 Call ID: call_cad80716089345b49fa56e01
  Args:
    query: What is the training objective for the discriminator in a GAN?
================================= Tool Message =================================
Name: retrieve_context


===============